In [1]:
import os
import numpy as np
import pandas as pd
import pickle

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, log_loss, brier_score_loss
from sklearn.model_selection import ShuffleSplit, GridSearchCV

pd.set_option("display.max_rows", None, "display.max_columns", None)

RANDOM_STATE = 42

## Getting the data ready

In [2]:
def get_datasets(span = 5):
    path = os.path.abspath(f'../../data/dataset/women/{span}span_training_set.csv')
    training_df = pd.read_csv(path)

    path = os.path.abspath(f'../../data/dataset/women/{span}span_testing_set.csv')
    testing_df = pd.read_csv(path)

    train_true, test_true = training_df.pop('Win'), testing_df.pop('Win')

    print(f'{len(training_df)} train examples')
    print(f'{len(testing_df)} test examples')

    return training_df, testing_df, train_true, test_true

## Training the model

In [3]:
def train_model(estimator, param_grid, training_df, train_true, n_splits=3):
    # Wrap estimator in a Pipeline with StandardScaler
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', estimator)
    ])

    # Prefix param_grid keys with 'model__' for the pipeline
    pipe_param_grid = {f'model__{k}': v for k, v in param_grid.items()}

    # Use log_loss scoring to optimize probability quality (what the ensemble actually uses)
    grid_search = GridSearchCV(
        pipe, pipe_param_grid,
        cv=ShuffleSplit(n_splits, random_state=RANDOM_STATE),
        scoring='neg_log_loss',
        verbose=5,
        n_jobs=-1
    )

    grid_search.fit(training_df, train_true)

    best_params = {k.replace('model__', ''): v for k, v in grid_search.best_params_.items()}
    print(f"Best hyperparameters: {best_params}")
    print(f"Best CV log_loss: {-grid_search.best_score_:.4f}")

    return grid_search.best_estimator_

## Testing the model

In [4]:
def test_model(clf, testing_df, test_true):
    y_pred = clf.predict(testing_df)
    y_proba = clf.predict_proba(testing_df)[:, 1]

    accuracy = accuracy_score(test_true, y_pred)
    logloss = log_loss(test_true, y_proba)
    brier = brier_score_loss(test_true, y_proba)

    print(f"Accuracy: {(accuracy*100):.2f}%")
    print(f"Log Loss: {logloss:.4f}")
    print(f"Brier Score: {brier:.4f}")

    print("\nClassification Report:")
    print(classification_report(test_true, y_pred))

    print("Confusion Matrix:")
    print(confusion_matrix(test_true, y_pred))

## Saving the models

In [5]:
def save_model(model, filename):
    path = os.path.abspath(f'../model/womens/{filename}')
    pickle.dump(model, open(path, 'wb'))

## Constant Variables

In [6]:
SPANS = [3, 5, 7]

In [7]:
def train_test_save(estimator, param_grid, filename, spans=[3, 5, 7], n_splits=3):
    for span in spans:
        print(f'\n{"="*50}')
        print(f'Span: {span}')
        print(f'{"="*50}')
        
        training_df, testing_df, train_true, test_true = get_datasets(span)
        clf = train_model(estimator, param_grid, training_df, train_true, n_splits)
        test_model(clf, testing_df, test_true)
        save_model(clf, f'{span}span_{filename}')

# Logistic Regression

In [8]:
# Define the Logistic Regression model
logreg_model = LogisticRegression(random_state=RANDOM_STATE)

# Define the hyperparameter grid
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['saga'],
    'max_iter': [10000]
}

filename = 'logistic_regression_model.pkl'

train_test_save(logreg_model, param_grid, filename)


Span: 3
17184 train examples
7365 test examples
Fitting 3 folds for each of 12 candidates, totalling 36 fits
[CV 3/3] END model__C=0.001, model__max_iter=10000, model__penalty=l1, model__solver=saga;, score=-0.575 total time=   2.7s
[CV 1/3] END model__C=0.001, model__max_iter=10000, model__penalty=l2, model__solver=saga;, score=-0.506 total time=   3.1s
[CV 3/3] END model__C=0.001, model__max_iter=10000, model__penalty=l2, model__solver=saga;, score=-0.512 total time=   3.0s
[CV 2/3] END model__C=0.001, model__max_iter=10000, model__penalty=l2, model__solver=saga;, score=-0.527 total time=   3.3s
[CV 1/3] END model__C=0.001, model__max_iter=10000, model__penalty=l1, model__solver=saga;, score=-0.567 total time=   2.8s
[CV 2/3] END model__C=0.001, model__max_iter=10000, model__penalty=l1, model__solver=saga;, score=-0.577 total time=   3.1s
[CV 3/3] END model__C=0.01, model__max_iter=10000, model__penalty=l2, model__solver=saga;, score=-0.511 total time=   5.8s
[CV 1/3] END model__C=0

# Support Vector Machine

In [9]:
# Define the SVM hyperparameter grid
param_grid = {
    'C': [0.1, 1],
    'kernel': ['rbf', 'linear'],
    'gamma': ['scale', 'auto']
}

filename = 'support_vector_machine_model.pkl'

# Custom loop: grid search WITHOUT probability (5x faster), then retrain best with probability=True
for span in [3, 5, 7]:
    print(f'\n{"="*50}')
    print(f'Span: {span}')
    print(f'{"="*50}')

    training_df, testing_df, train_true, test_true = get_datasets(span)

    # Phase 1: Fast grid search (no Platt scaling)
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(random_state=RANDOM_STATE))  # probability=False (default)
    ])
    pipe_param_grid = {f'model__{k}': v for k, v in param_grid.items()}

    grid_search = GridSearchCV(
        pipe, pipe_param_grid,
        cv=ShuffleSplit(3, random_state=RANDOM_STATE),
        scoring='accuracy',  # can't use log_loss without predict_proba
        verbose=5,
        n_jobs=-1
    )
    grid_search.fit(training_df, train_true)

    best_params = {k.replace('model__', ''): v for k, v in grid_search.best_params_.items()}
    print(f"Best hyperparameters: {best_params}")
    print(f"Best CV accuracy: {grid_search.best_score_:.4f}")

    # Phase 2: Retrain once with probability=True using best params
    print("Retraining with probability=True for predict_proba support...")
    best_params['probability'] = True
    best_params['random_state'] = RANDOM_STATE
    final_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(**best_params))
    ])
    final_pipe.fit(training_df, train_true)

    test_model(final_pipe, testing_df, test_true)
    save_model(final_pipe, f'{span}span_{filename}')


Span: 3
17184 train examples
7365 test examples
Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV 3/3] END model__C=0.1, model__gamma=scale, model__kernel=rbf;, score=0.755 total time= 6.7min
[CV 1/3] END model__C=0.1, model__gamma=auto, model__kernel=rbf;, score=0.749 total time= 6.8min
[CV 2/3] END model__C=1, model__gamma=scale, model__kernel=rbf;, score=0.721 total time= 6.8min
[CV 2/3] END model__C=0.1, model__gamma=auto, model__kernel=rbf;, score=0.725 total time= 6.9min
[CV 2/3] END model__C=0.1, model__gamma=scale, model__kernel=rbf;, score=0.725 total time= 7.0min
[CV 1/3] END model__C=1, model__gamma=scale, model__kernel=rbf;, score=0.756 total time= 7.0min
[CV 1/3] END model__C=0.1, model__gamma=scale, model__kernel=rbf;, score=0.749 total time= 7.3min
[CV 3/3] END model__C=0.1, model__gamma=auto, model__kernel=rbf;, score=0.755 total time= 7.8min
[CV 3/3] END model__C=1, model__gamma=scale, model__kernel=rbf;, score=0.760 total time= 8.2min
[CV 1/3] END model

# K-Nearest Neighbors (KNN)

In [10]:
# Define the KNN model
knn = KNeighborsClassifier()

# Define the hyperparameter grid
param_grid = {
    'n_neighbors': [3, 5, 7, 11, 15],
    'weights': ['uniform', 'distance'],
    'p': [1, 2],
    'leaf_size': [20, 30, 50]
}

filename = 'knn_model.pkl'

train_test_save(knn, param_grid, filename)


Span: 3
17184 train examples
7365 test examples
Fitting 3 folds for each of 60 candidates, totalling 180 fits
[CV 2/3] END model__leaf_size=20, model__n_neighbors=3, model__p=2, model__weights=distance;, score=-3.623 total time=   2.1s
[CV 1/3] END model__leaf_size=20, model__n_neighbors=3, model__p=2, model__weights=uniform;, score=-3.419 total time=   2.3s
[CV 1/3] END model__leaf_size=20, model__n_neighbors=3, model__p=2, model__weights=distance;, score=-3.419 total time=   2.3s
[CV 2/3] END model__leaf_size=20, model__n_neighbors=3, model__p=2, model__weights=uniform;, score=-3.623 total time=   2.4s
[CV 3/3] END model__leaf_size=20, model__n_neighbors=3, model__p=2, model__weights=distance;, score=-3.669 total time=   2.1s
[CV 3/3] END model__leaf_size=20, model__n_neighbors=3, model__p=2, model__weights=uniform;, score=-3.670 total time=   2.2s
[CV 1/3] END model__leaf_size=20, model__n_neighbors=5, model__p=2, model__weights=uniform;, score=-1.491 total time=   1.2s
[CV 2/3] EN

# Random Forests

In [11]:
# Define the Random Forest model
rfc = RandomForestClassifier(random_state=RANDOM_STATE)

# Define the hyperparameter grid
param_grid = {
    'n_estimators': [200, 500],
    'max_features': ['sqrt', 'log2'],
    'max_depth': [8, 12, 20, None],
    'criterion': ['gini', 'entropy']
}

filename = 'random_forest.pkl'

train_test_save(rfc, param_grid, filename)


Span: 3
17184 train examples
7365 test examples
Fitting 3 folds for each of 32 candidates, totalling 96 fits
[CV 2/3] END model__criterion=gini, model__max_depth=8, model__max_features=log2, model__n_estimators=200;, score=-0.560 total time=  14.9s
[CV 1/3] END model__criterion=gini, model__max_depth=8, model__max_features=log2, model__n_estimators=200;, score=-0.543 total time=  15.0s
[CV 3/3] END model__criterion=gini, model__max_depth=8, model__max_features=log2, model__n_estimators=200;, score=-0.549 total time=  15.7s
[CV 3/3] END model__criterion=gini, model__max_depth=8, model__max_features=sqrt, model__n_estimators=200;, score=-0.539 total time=  30.7s
[CV 1/3] END model__criterion=gini, model__max_depth=8, model__max_features=sqrt, model__n_estimators=200;, score=-0.530 total time=  31.2s
[CV 2/3] END model__criterion=gini, model__max_depth=8, model__max_features=sqrt, model__n_estimators=200;, score=-0.549 total time=  31.2s
[CV 1/3] END model__criterion=gini, model__max_dep

# Gradient Boosting

In [12]:
# Define the Gradient Boosting model
gbc = GradientBoostingClassifier(random_state=RANDOM_STATE)

# Define the hyperparameter grid
param_grid = {
    'loss': ['log_loss', 'exponential'],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 8],
    'max_features': ['log2', 'sqrt'],
    'n_estimators': [200, 500]
}

filename = 'gradient_boosting.pkl'

train_test_save(gbc, param_grid, filename)


Span: 3
17184 train examples
7365 test examples
Fitting 3 folds for each of 72 candidates, totalling 216 fits
[CV 1/3] END model__learning_rate=0.01, model__loss=log_loss, model__max_depth=3, model__max_features=log2, model__n_estimators=200;, score=-0.576 total time=   8.8s
[CV 2/3] END model__learning_rate=0.01, model__loss=log_loss, model__max_depth=3, model__max_features=log2, model__n_estimators=200;, score=-0.589 total time=   9.6s
[CV 3/3] END model__learning_rate=0.01, model__loss=log_loss, model__max_depth=3, model__max_features=log2, model__n_estimators=200;, score=-0.581 total time=   9.7s
[CV 3/3] END model__learning_rate=0.01, model__loss=log_loss, model__max_depth=5, model__max_features=log2, model__n_estimators=200;, score=-0.554 total time=  14.7s
[CV 1/3] END model__learning_rate=0.01, model__loss=log_loss, model__max_depth=5, model__max_features=log2, model__n_estimators=200;, score=-0.549 total time=  15.1s
[CV 2/3] END model__learning_rate=0.01, model__loss=log_los

# Multilayer Perceptron

In [13]:
# Define the MLP model
mlp_model = MLPClassifier(random_state=RANDOM_STATE, early_stopping=True, max_iter=500)

# Define the hyperparameter grid
param_grid = {
    'hidden_layer_sizes': [(128, 64), (256, 128), (354, 177), (256,)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam'],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate': ['adaptive']
}

filename = 'multilayer_perceptron.pkl'

train_test_save(mlp_model, param_grid, filename)


Span: 3
17184 train examples
7365 test examples
Fitting 3 folds for each of 24 candidates, totalling 72 fits
[CV 2/3] END model__activation=relu, model__alpha=0.001, model__hidden_layer_sizes=(128, 64), model__learning_rate=adaptive, model__solver=adam;, score=-0.541 total time=  16.2s
[CV 3/3] END model__activation=relu, model__alpha=0.0001, model__hidden_layer_sizes=(128, 64), model__learning_rate=adaptive, model__solver=adam;, score=-0.525 total time=  18.5s
[CV 2/3] END model__activation=relu, model__alpha=0.0001, model__hidden_layer_sizes=(128, 64), model__learning_rate=adaptive, model__solver=adam;, score=-0.541 total time=  19.5s
[CV 1/3] END model__activation=relu, model__alpha=0.0001, model__hidden_layer_sizes=(128, 64), model__learning_rate=adaptive, model__solver=adam;, score=-0.525 total time=  19.5s
[CV 1/3] END model__activation=relu, model__alpha=0.001, model__hidden_layer_sizes=(128, 64), model__learning_rate=adaptive, model__solver=adam;, score=-0.532 total time=  21.

# XGBoost

In [8]:
# Define the XGBoost model
xgb_model = XGBClassifier(
    random_state=RANDOM_STATE,
    eval_metric='logloss'
)

# Define the hyperparameter grid (trimmed: 72 candidates)
param_grid = {
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 8],
    'n_estimators': [200, 500],
    'subsample': [0.8],
    'colsample_bytree': [0.8, 1.0],
    'reg_alpha': [0],
    'reg_lambda': [1, 5]
}

filename = 'xgboost.pkl'

train_test_save(xgb_model, param_grid, filename)


Span: 3
17184 train examples
7365 test examples
Fitting 3 folds for each of 72 candidates, totalling 216 fits
[CV 2/3] END model__colsample_bytree=0.8, model__learning_rate=0.01, model__max_depth=3, model__n_estimators=200, model__reg_alpha=0, model__reg_lambda=5, model__subsample=0.8;, score=-0.560 total time=  25.2s
[CV 3/3] END model__colsample_bytree=0.8, model__learning_rate=0.01, model__max_depth=3, model__n_estimators=200, model__reg_alpha=0, model__reg_lambda=5, model__subsample=0.8;, score=-0.553 total time=  28.6s
[CV 2/3] END model__colsample_bytree=0.8, model__learning_rate=0.01, model__max_depth=3, model__n_estimators=200, model__reg_alpha=0, model__reg_lambda=1, model__subsample=0.8;, score=-0.560 total time=  29.1s
[CV 1/3] END model__colsample_bytree=0.8, model__learning_rate=0.01, model__max_depth=3, model__n_estimators=200, model__reg_alpha=0, model__reg_lambda=5, model__subsample=0.8;, score=-0.547 total time=  28.8s
[CV 3/3] END model__colsample_bytree=0.8, model__

# LightGBM

In [9]:
# Define the LightGBM model
lgbm_model = LGBMClassifier(
    random_state=RANDOM_STATE,
    verbose=-1
)

# Define the hyperparameter grid (trimmed: 72 candidates)
param_grid = {
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 8],
    'n_estimators': [200, 500],
    'num_leaves': [31],
    'subsample': [0.8],
    'colsample_bytree': [0.8, 1.0],
    'reg_alpha': [0],
    'reg_lambda': [1, 5]
}

filename = 'lightgbm.pkl'

train_test_save(lgbm_model, param_grid, filename)


Span: 3
17184 train examples
7365 test examples
Fitting 3 folds for each of 72 candidates, totalling 216 fits
[CV 3/3] END model__colsample_bytree=0.8, model__learning_rate=0.01, model__max_depth=3, model__n_estimators=200, model__num_leaves=31, model__reg_alpha=0, model__reg_lambda=1, model__subsample=0.8;, score=-0.554 total time= 2.1min
[CV 1/3] END model__colsample_bytree=0.8, model__learning_rate=0.01, model__max_depth=3, model__n_estimators=200, model__num_leaves=31, model__reg_alpha=0, model__reg_lambda=5, model__subsample=0.8;, score=-0.548 total time= 2.2min
[CV 2/3] END model__colsample_bytree=0.8, model__learning_rate=0.01, model__max_depth=3, model__n_estimators=200, model__num_leaves=31, model__reg_alpha=0, model__reg_lambda=1, model__subsample=0.8;, score=-0.560 total time= 2.2min
[CV 2/3] END model__colsample_bytree=0.8, model__learning_rate=0.01, model__max_depth=3, model__n_estimators=200, model__num_leaves=31, model__reg_alpha=0, model__reg_lambda=5, model__subsample

# Feature Importance Analysis

In [14]:
# Extract feature importances from tree-based models (span=5 as representative)
span = 5
training_df, testing_df, train_true, test_true = get_datasets(span)
feature_names = training_df.columns.tolist()

# Load the saved Random Forest and Gradient Boosting models for this span
rf_model = pickle.load(open(os.path.abspath(f'../model/womens/{span}span_random_forest.pkl'), 'rb'))
gb_model = pickle.load(open(os.path.abspath(f'../model/womens/{span}span_gradient_boosting.pkl'), 'rb'))

# Feature importance from Random Forest (pipeline: access the model step)
rf_importances = rf_model.named_steps['model'].feature_importances_
rf_importance_df = pd.DataFrame({'feature': feature_names, 'importance': rf_importances}).sort_values('importance', ascending=False)

# Feature importance from Gradient Boosting
gb_importances = gb_model.named_steps['model'].feature_importances_
gb_importance_df = pd.DataFrame({'feature': feature_names, 'importance': gb_importances}).sort_values('importance', ascending=False)

print("=== Random Forest: Top 30 Features ===")
print(rf_importance_df.head(30).to_string(index=False))

print("\n=== Gradient Boosting: Top 30 Features ===")
print(gb_importance_df.head(30).to_string(index=False))

# How much do defensive features contribute?
def_features = [f for f in feature_names if 'def_' in f]
rf_def_total = rf_importance_df[rf_importance_df['feature'].isin(def_features)]['importance'].sum()
gb_def_total = gb_importance_df[gb_importance_df['feature'].isin(def_features)]['importance'].sum()
print(f"\n=== Defensive Feature Contribution ===")
print(f"Random Forest:     {rf_def_total:.4f} ({rf_def_total*100:.1f}% of total)")
print(f"Gradient Boosting: {gb_def_total:.4f} ({gb_def_total*100:.1f}% of total)")

15584 train examples
6680 test examples
=== Random Forest: Top 30 Features ===
     feature  importance
    ORtg_CMA    0.015685
opp_ORtg_CMA    0.013292
opp_ORtg_EMA    0.010056
opp_ORtg_SMA    0.009911
    ORtg_EMA    0.009658
    ORtg_SMA    0.009476
      FG_CMA    0.008594
    DRtg_CMA    0.008347
    TOV%_CMA    0.007841
opp_DRtg_CMA    0.007748
  opp_FG_CMA    0.007657
 opp_TS%_CMA    0.007134
opp_TOV%_CMA    0.006812
 opp_FG%_CMA    0.006644
     TS%_CMA    0.006630
     FG%_CMA    0.006604
opp_eFG%_CMA    0.006276
    eFG%_CMA    0.006175
     AST_CMA    0.005994
     2P%_CMA    0.005844
 opp_2P%_CMA    0.005460
 opp_AST_CMA    0.005439
    TRB%_CMA    0.005115
  opp_FG_SMA    0.005015
 def_AST_CMA    0.004989
    DRtg_EMA    0.004947
     TOV_CMA    0.004725
opp_TRB%_CMA    0.004657
      FG_EMA    0.004633
 opp_TOV_CMA    0.004593

=== Gradient Boosting: Top 30 Features ===
         feature  importance
        ORtg_SMA    0.058007
      opp_FG_CMA    0.041085
        ORtg_CM

In [ ]:
# Extract all model metrics for research paper
import json

model_files = {
    'Logistic Regression': 'logistic_regression_model.pkl',
    'SVM': 'support_vector_machine_model.pkl',
    'KNN': 'knn_model.pkl',
    'Random Forest': 'random_forest.pkl',
    'Gradient Boosting': 'gradient_boosting.pkl',
    'MLP': 'multilayer_perceptron.pkl',
    'XGBoost': 'xgboost.pkl',
    'LightGBM': 'lightgbm.pkl'
}

womens_results = {}
for span in [3, 5, 7]:
    training_df, testing_df, train_true, test_true = get_datasets(span)
    womens_results[span] = {}
    for name, fname in model_files.items():
        model = pickle.load(open(os.path.abspath(f'../model/womens/{span}span_{fname}'), 'rb'))
        y_pred = model.predict(testing_df)
        y_proba = model.predict_proba(testing_df)[:, 1]
        acc = accuracy_score(test_true, y_pred)
        ll = log_loss(test_true, y_proba)
        bs = brier_score_loss(test_true, y_proba)
        womens_results[span][name] = {'accuracy': round(acc*100, 2), 'log_loss': round(ll, 4), 'brier_score': round(bs, 4)}

# Print summary table
print("=" * 80)
print("WOMEN'S BASKETBALL - MODEL RESULTS SUMMARY")
print("=" * 80)
for span in [3, 5, 7]:
    print(f"\n--- Span {span} ---")
    print(f"{'Model':<25} {'Accuracy':>10} {'Log Loss':>10} {'Brier':>10}")
    print("-" * 55)
    for name in model_files:
        r = womens_results[span][name]
        print(f"{name:<25} {r['accuracy']:>9.2f}% {r['log_loss']:>10.4f} {r['brier_score']:>10.4f}")

# Average across spans
print(f"\n--- Average Across Spans ---")
print(f"{'Model':<25} {'Accuracy':>10} {'Log Loss':>10} {'Brier':>10}")
print("-" * 55)
for name in model_files:
    avg_acc = np.mean([womens_results[s][name]['accuracy'] for s in [3,5,7]])
    avg_ll = np.mean([womens_results[s][name]['log_loss'] for s in [3,5,7]])
    avg_bs = np.mean([womens_results[s][name]['brier_score'] for s in [3,5,7]])
    print(f"{name:<25} {avg_acc:>9.2f}% {avg_ll:>10.4f} {avg_bs:>10.4f}")

print("\n\nJSON_DATA_START")
print(json.dumps(womens_results, indent=2))
print("JSON_DATA_END")

17184 train examples
7365 test examples
15584 train examples
6680 test examples
13997 train examples
6000 test examples
WOMEN'S BASKETBALL - MODEL RESULTS SUMMARY

--- Span 3 ---
Model                       Accuracy   Log Loss      Brier
-------------------------------------------------------
Logistic Regression           73.17%     0.5207     0.1747
SVM                           73.13%     0.5235     0.1758
KNN                           69.69%     0.6216     0.1943
Random Forest                 72.78%     0.5381     0.1811
Gradient Boosting             72.61%     0.5279     0.1778
MLP                           72.86%     0.5302     0.1784

--- Span 5 ---
Model                       Accuracy   Log Loss      Brier
-------------------------------------------------------
Logistic Regression           73.56%     0.5180     0.1737
SVM                           73.40%     0.5239     0.1760
KNN                           70.64%     0.6116     0.1899
Random Forest                 73.34%     0.5